In [1]:
import os
os.environ["OPENAI_API_KEY"] = "ollama"
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"

from pathlib import Path
import json
import re
import tempfile
from typing import Any, Dict, List

import pandas as pd
from openai import OpenAI
from lida import Manager, llm


In [2]:
def flatten_json(obj, parent="", sep="_"):
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{parent}{sep}{k}" if parent else k
            if isinstance(v, dict):
                out.update(flatten_json(v, key, sep))
            else:
                out[key] = v
    return out


def load_any_df(path: str, sample_rows=5000):
    p = Path(path)
    ext = p.suffix.lower()

    if ext in [".csv", ".tsv"]:
        return pd.read_csv(p, nrows=sample_rows)

    if ext in [".log", ".ndjson", ".jsonl"]:
        rows = []
        with open(p, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= sample_rows:
                    break
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    rows.append(flatten_json(obj))
                except:
                    continue
        return pd.DataFrame(rows)

    return pd.read_csv(p, nrows=sample_rows)


In [3]:
def sanitize_mapping(mapping: dict, df: pd.DataFrame):
    cols = set(df.columns)

    if not isinstance(mapping, dict):
        mapping = {}

    mapping.setdefault("x", None)
    mapping.setdefault("y", None)
    mapping.setdefault("chart", "auto")
    if mapping.get("filter") is None:
        mapping["filter"] = []

    for k in ["x","y"]:
        if mapping.get(k) not in cols:
            mapping[k] = None

    valid = []
    for f in (mapping.get("filter") or []):
        if isinstance(f, dict) and f.get("column") in cols:
            valid.append(f)
    mapping["filter"] = valid

    return mapping


def apply_filters(df, mapping):
    out = df.copy()
    for f in mapping.get("filter", []):
        col = f["column"]
        op = f["op"]
        val = f["value"]

        if col not in out.columns:
            continue

        s = out[col]

        if s.dropna().isin([True, False]).all():
            if op == "==":
                out = out[s == bool(val)]
            elif op == "!=":
                out = out[s != bool(val)]
            continue

        if op == "==":
            out = out[s == val]
        elif op == "!=":
            out = out[s != val]

    return out


In [4]:
def llm_map_columns(prompt: str, df: pd.DataFrame):
    schema = list(df.columns)

    client = OpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )

    system = """Return ONLY valid JSON:
{
 "x": string|null,
 "y": string|null,
 "chart": "bar"|"pie"|"scatter"|"line"|"auto",
 "filter": []
}
Use only column names from the schema."""

    msg = f"Schema: {schema}\nUser request: {prompt}"

    resp = client.chat.completions.create(
        model="qwen2.5:3b",
        messages=[
            {"role":"system","content":system},
            {"role":"user","content":msg}
        ],
        temperature=0.1
    )

    text = resp.choices[0].message.content.strip()

    try:
        mapping = json.loads(text)
    except:
        mapping = {"x":None,"y":None,"chart":"auto","filter":[]}

    return mapping


In [5]:
def prepare_count_df(df, group_col):
    out = df[group_col].value_counts().reset_index()
    out.columns = [group_col, "count"]
    return out


In [8]:
def run_lida_clean(file, prompt, out="out.png"):

    df = load_any_df(file)

    mapping = llm_map_columns(prompt, df)
    mapping = sanitize_mapping(mapping, df)

    print("\nMAPPING:", mapping)

    df2 = apply_filters(df, mapping)

    # 🔥 auto detect "nombre"
    if "nombre" in prompt.lower() or "count" in prompt.lower():
        if mapping["x"] in df2.columns:
            df_plot = prepare_count_df(df2, mapping["x"])
            mapping["y"] = "count"
            mapping["chart"] = "bar"
        else:
            df_plot = df2
    else:
        df_plot = df2

    print("\nDF_PLOT HEAD:")
    print(df_plot.head())

    with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False) as tmp:
        df_plot.to_csv(tmp.name, index=False)
        tmp_path = tmp.name

    lida_mgr = Manager(text_gen=llm("openai", model="qwen2.5:3b"))
    summary = lida_mgr.summarize(tmp_path)

    goal = {
        "question": prompt,
        "visualization": f"Create a {mapping['chart']} chart using matplotlib. Use x={mapping['x']} and y={mapping['y']}.",
        "rationale": "Mapping from prompt to columns."
    }

    charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")

    if charts:

        chart_obj = charts[0]

        raster = getattr(chart_obj, "raster", None)

        if isinstance(raster, (bytes, bytearray)):
            with open(out, "wb") as f:
              f.write(raster)

        elif isinstance(raster, str):
         import base64
         if raster.startswith("data:image"):
             raster = raster.split(",", 1)[1]
         with open(out, "wb") as f:
            f.write(base64.b64decode(raster))

        else:
            print("No raster found in LIDA response.")

        print("\nGraph generated:", out)

    else:
        print("No chart generated.")


In [9]:
run_lida_clean(
    file="data/lidata.log",
    prompt="Bar chart du nombre d'entités par EntityType"
)



MAPPING: {'x': 'EntityType', 'y': None, 'chart': 'bar', 'filter': []}

DF_PLOT HEAD:
         EntityType  count
0  1:1:222:1:12:0:0      2
1    1:1:71:2:5:0:1      1
```python
import matplotlib.pyplot as plt

def plot(data: pd.DataFrame):
    EntityType_counts = data['count'].value_counts().reset_index()
    EntityType_counts.columns = ['EntityType', 'count']
    
    ax = EntityType_counts.plot(x='EntityType', y='count', kind='bar')
    ax.set_xlabel('EntityType')
    ax.set_ylabel('Count')
    plt.title('Bar chart du nombre d\'entités par EntityType', wrap=True)
    return plt

chart = plot(data)
```
****
 cannot insert count, already exists
No chart generated.
